In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. Load the datasets
collision_df = pd.read_csv('final_merged_cleaned_data.csv')
mapping_df = pd.read_csv('metadata_images_mapping.csv')
features_df = pd.read_csv('processed_features_conf_0.7.csv')

# Clean missing values
mapping_df = mapping_df.dropna(subset=['image_blob_name', 'lat', 'lng'])
features_df = features_df.dropna(subset=['image_name'])

# 2. Link Features to Coordinates
# Extract the base filename (e.g., '4411_0.jpg') from the full blob path
mapping_df['image_filename'] = mapping_df['image_blob_name'].astype(str).apply(lambda x: x.split('/')[-1])

# Merge visual features with their lat/lng coordinates
visual_data = pd.merge(features_df, mapping_df[['image_filename', 'lat', 'lng']],
                       left_on='image_name', right_on='image_filename')

# Average the features for the 4 image angles at each unique location
visual_features_per_loc = visual_data.drop(columns=['image_name', 'image_filename']).groupby(['lat', 'lng']).mean().reset_index()

# 3. Spatial Search (100m Radius)
# We scale Longitude by cos(Lat) to approximate Euclidean distance in meters
mean_lat = np.radians(51.5) # Approximate latitude for London
lng_scale = np.cos(mean_lat)

visual_coords = visual_features_per_loc[['lat', 'lng']].values
visual_coords_scaled = visual_coords.copy()
visual_coords_scaled[:, 1] *= lng_scale

collision_coords = collision_df[['latitude', 'longitude']].values
collision_coords_scaled = collision_coords.copy()
collision_coords_scaled[:, 1] *= lng_scale

# Build spatial index
tree = cKDTree(visual_coords_scaled)

# 100 meters expressed in degrees latitude
radius_deg = 100 / 111320.0

# Find the nearest visual features within 100m for every accident
distances, indices = tree.query(collision_coords_scaled, distance_upper_bound=radius_deg)

# 4. Merge Data
matched_mask = indices < len(visual_features_per_loc)
feature_cols = visual_features_per_loc.columns.drop(['lat', 'lng'])

# Prepare empty columns for features
matched_feature_values = np.full((len(collision_df), len(feature_cols)), np.nan)
# Fill rows where a match was found
matched_feature_values[matched_mask] = visual_features_per_loc.iloc[indices[matched_mask]][feature_cols].values

# Create final DataFrame
features_to_add = pd.DataFrame(matched_feature_values, columns=feature_cols, index=collision_df.index)
enriched_df = pd.concat([collision_df, features_to_add], axis=1)

# 5. Save
enriched_df.to_csv('enriched_collision_data_100m.csv', index=False)

print(f"Total collisions: {len(collision_df)}")
print(f"Matched with features: {matched_mask.sum()} ({matched_mask.mean()*100:.2f}%)")